# Hello GPU

A tiny notebook to check that KaggleFleet can run code on a Kaggle accelerator.

```bash
kfleet run push examples/hello_gpu.ipynb --watch
```

It prints the hardware it got and writes `/kaggle/working/hello.json`, which `--watch` then downloads to `./outputs/<slug>/`.

In [ ]:
import json, platform, subprocess, time

def gpu_info():
    try:
        return subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                              capture_output=True, text=True, timeout=30).stdout.strip() or 'no GPU'
    except FileNotFoundError:
        return 'no GPU (nvidia-smi not found)'

info = {'python': platform.python_version(), 'gpu': gpu_info(), 'finished_at': time.strftime('%Y-%m-%d %H:%M:%S')}
print(json.dumps(info, indent=2))

In [ ]:
try:
    import torch
    if torch.cuda.is_available():
        x = torch.randn(4096, 4096, device='cuda')
        torch.cuda.synchronize(); t = time.time()
        for _ in range(10):
            y = x @ x
        torch.cuda.synchronize()
        info['matmul_10x_4096_seconds'] = round(time.time() - t, 3)
except ImportError:
    pass

with open('/kaggle/working/hello.json', 'w') as f:
    json.dump(info, f, indent=2)
info